# Staffing by Archetype

This notebook uses the archetype labels from `01_rules_based.ipynb` to explore staffing
structure across the practice portfolio. The goal is twofold:

1. **Define required staffing levels** — understand the typical headcount and hour
   profile for each archetype so that resourcing decisions are archetype-aware rather
   than one-size-fits-all.

2. **Enable career mobility** — understand which roles exist at each archetype and
   how a career can progress across the size and model dimensions of the framework.

## Roles covered

| Role | Type | Column |
|------|------|--------|
| Dentist | Clinical | `position_dentist` / `contractualhours_dentist` |
| Dental Nurse | Clinical | `position_dental_nurse` / `contractualhours_dental_nurse` |
| Hygienist | Clinical (specialist) | `position_hygienist` / `contractualhours_hygienist` |
| Receptionist | Support | `position_receptionist` / `contractualhours_receptionist` |
| Practice Manager | Support | `position_practice_manager` / `contractualhours_practice_manager` |

> **Note:** Run `01_rules_based.ipynb` first to generate `archetypes_rules.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#55A868'
RED    = '#C44E52'
PURPLE = '#8172B2'
GREY   = '#8C8C8C'

ROLE_COLORS = {
    'Dentist':           '#4C72B0',
    'Dental Nurse':      '#DD8452',
    'Hygienist':         '#55A868',
    'Receptionist':      '#C44E52',
    'Practice Manager':  '#8172B2',
}
ROLES        = list(ROLE_COLORS.keys())
ROLE_POS_COLS  = ['position_dentist','position_dental_nurse','position_hygienist',
                  'position_receptionist','position_practice_manager']
ROLE_HRS_COLS  = ['contractualhours_dentist','contractualhours_dental_nurse',
                  'contractualhours_hygienist','contractualhours_receptionist',
                  'contractualhours_practice_manager']

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

SIZE_ORDER  = ['Small/Foundation', 'Medium/Core', 'Large/Advanced', 'Flagship']
MODEL_ORDER = ['NHS Led', 'Balanced Mixed', 'Private Led Mixed', 'Specialist/Referral Hub']

# Load and merge
master = pd.read_csv('master.csv')
arc    = pd.read_csv('archetypes_rules.csv')[['practicekey','archetype_size','archetype_model','archetype_rules']]
df = master.merge(arc, on='practicekey', how='inner')

df['archetype_size']  = pd.Categorical(df['archetype_size'],  categories=SIZE_ORDER,  ordered=True)
df['archetype_model'] = pd.Categorical(df['archetype_model'], categories=MODEL_ORDER, ordered=True)

# Derived
df['total_headcount']    = df[ROLE_POS_COLS].sum(axis=1)
df['clinical_headcount'] = df[['position_dentist','position_dental_nurse','position_hygienist']].sum(axis=1)
df['support_headcount']  = df[['position_receptionist','position_practice_manager']].sum(axis=1)
df['total_hours']        = df[ROLE_HRS_COLS].sum(axis=1)
df['clinical_hours']     = df[['contractualhours_dentist','contractualhours_dental_nurse',
                                'contractualhours_hygienist']].sum(axis=1)
df['nurse_dentist_ratio']= df['position_dental_nurse'] / df['position_dentist'].replace(0, np.nan)
df['has_hygienist']      = df['position_hygienist'] > 0

print(f'Loaded {len(df):,} practices across {df["archetype_rules"].nunique()} archetype cells')

---
## 1 · Headcount composition by size band

As practices grow from Small/Foundation to Flagship, the absolute headcount
increases — but the *mix* of roles matters for understanding career opportunity.
The stacked bars show the mean headcount per role at each size band.

In [ ]:
avg_by_size = (
    df.groupby('archetype_size', observed=True)[ROLE_POS_COLS]
    .mean()
    .reindex(SIZE_ORDER)
)
avg_by_size.columns = ROLES

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('1 · Headcount Composition by Size Band', fontsize=13, fontweight='bold')

# Stacked bar
bottom = np.zeros(len(SIZE_ORDER))
x = np.arange(len(SIZE_ORDER))
for role in ROLES:
    vals = avg_by_size[role].values
    axes[0].bar(x, vals, bottom=bottom, color=ROLE_COLORS[role], label=role, edgecolor='white')
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.3:
            axes[0].text(xi, b + v/2, f'{v:.1f}', ha='center', va='center',
                         fontsize=8, color='white', fontweight='bold')
    bottom += vals
axes[0].set_xticks(x)
axes[0].set_xticklabels(SIZE_ORDER, rotation=15, ha='right')
axes[0].set_ylabel('Average headcount')
axes[0].set_title('Average headcount per role (stacked)')
axes[0].legend(loc='upper left', fontsize=9)

# Clinical vs support ratio
ratio = df.groupby('archetype_size', observed=True)[['clinical_headcount','support_headcount']].mean().reindex(SIZE_ORDER)
ratio.plot(kind='bar', ax=axes[1], color=[GREEN, ORANGE], edgecolor='white', width=0.6)
axes[1].set_xticklabels(SIZE_ORDER, rotation=15, ha='right')
axes[1].set_ylabel('Average headcount')
axes[1].set_title('Clinical vs support headcount')
axes[1].legend(['Clinical', 'Support'], fontsize=9)

plt.tight_layout()
plt.show()

print('Average headcount by size band:')
display(avg_by_size.round(2))

---
## 2 · Headcount composition by model band

Model (NHS Led → Specialist/Referral Hub) drives a different kind of staffing
variation — particularly the presence of hygienists and the ratio of clinical to
support staff, which reflects the complexity of the patient mix.

In [ ]:
avg_by_model = (
    df.groupby('archetype_model', observed=True)[ROLE_POS_COLS]
    .mean()
    .reindex(MODEL_ORDER)
)
avg_by_model.columns = ROLES

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('2 · Headcount Composition by Model Band', fontsize=13, fontweight='bold')

bottom = np.zeros(len(MODEL_ORDER))
x = np.arange(len(MODEL_ORDER))
for role in ROLES:
    vals = avg_by_model[role].values
    axes[0].bar(x, vals, bottom=bottom, color=ROLE_COLORS[role], label=role, edgecolor='white')
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.2:
            axes[0].text(xi, b + v/2, f'{v:.1f}', ha='center', va='center',
                         fontsize=8, color='white', fontweight='bold')
    bottom += vals
axes[0].set_xticks(x)
axes[0].set_xticklabels(MODEL_ORDER, rotation=15, ha='right')
axes[0].set_ylabel('Average headcount')
axes[0].set_title('Average headcount per role (stacked)')
axes[0].legend(loc='upper left', fontsize=9)

# Hygienist penetration by model
hyg_pct = df.groupby('archetype_model', observed=True)['has_hygienist'].mean().reindex(MODEL_ORDER) * 100
bars = axes[1].bar(range(len(MODEL_ORDER)), hyg_pct.values, color=GREEN, edgecolor='white')
axes[1].bar_label(bars, fmt='%.1f%%', padding=3)
axes[1].set_xticks(range(len(MODEL_ORDER)))
axes[1].set_xticklabels(MODEL_ORDER, rotation=15, ha='right')
axes[1].set_ylabel('% of practices')
axes[1].set_title('Hygienist presence (% of practices)')
axes[1].set_ylim(0, 115)

plt.tight_layout()
plt.show()

---
## 3 · Contractual hours by role and archetype

Headcount tells one part of the story; contracted hours reveals the actual clinical
capacity committed. A practice may have a dentist on paper but at reduced hours.

In [ ]:
avg_hrs_size  = df.groupby('archetype_size',  observed=True)[ROLE_HRS_COLS].mean().reindex(SIZE_ORDER)
avg_hrs_model = df.groupby('archetype_model', observed=True)[ROLE_HRS_COLS].mean().reindex(MODEL_ORDER)
avg_hrs_size.columns  = ROLES
avg_hrs_model.columns = ROLES

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('3 · Contractual Hours by Role', fontsize=13, fontweight='bold')

for ax, data, labels, title in [
    (axes[0], avg_hrs_size,  SIZE_ORDER,  'By size band'),
    (axes[1], avg_hrs_model, MODEL_ORDER, 'By model band'),
]:
    bottom = np.zeros(len(labels))
    x = np.arange(len(labels))
    for role in ROLES:
        vals = data[role].values
        ax.bar(x, vals, bottom=bottom, color=ROLE_COLORS[role], label=role, edgecolor='white')
        for xi, (v, b) in enumerate(zip(vals, bottom)):
            if v > 5:
                ax.text(xi, b + v/2, f'{v:.0f}', ha='center', va='center',
                        fontsize=7, color='white', fontweight='bold')
        bottom += vals
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha='right')
    ax.set_ylabel('Average contracted hours per week')
    ax.set_title(title)
    if ax == axes[0]:
        ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

print('Average contracted hours by role — size band:')
display(avg_hrs_size.round(1))

---
## 4 · Key staffing ratios by archetype

Two ratios are especially important for resourcing decisions:

- **Nurse:Dentist ratio** — a ratio below 1:1 suggests either dental nurses are
  overstretched or some dentists lack dedicated chair-side support. A ratio above
  1.2:1 may indicate scope for dentist capacity growth without additional nursing.

- **Dentists per surgery** — reflects how intensively the physical estate is staffed.
  A ratio significantly below 1.0 suggests under-staffed surgeries (whitespace risk);
  above 1.0 suggests surgeries may be shared or scheduling is tight.

In [ ]:
df['dentists_per_surgery'] = df['position_dentist'] / df['numberofsurgeries'].replace(0, np.nan)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('4 · Key Staffing Ratios by Archetype', fontsize=13, fontweight='bold')

palette_size  = sns.color_palette('Blues_d',  4)
palette_model = sns.color_palette('Oranges_d', 4)

# Nurse:dentist by size
data = [df[df['archetype_size'] == s]['nurse_dentist_ratio'].dropna() for s in SIZE_ORDER]
bp = axes[0,0].boxplot(data, patch_artist=True, widths=0.5, notch=False,
                        medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'], palette_size):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[0,0].axhline(1.0, color=RED, lw=1.5, ls='--', label='1:1 benchmark')
axes[0,0].set_xticks(range(1, len(SIZE_ORDER)+1))
axes[0,0].set_xticklabels(SIZE_ORDER, rotation=12, ha='right')
axes[0,0].set_ylabel('Nurses per dentist')
axes[0,0].set_title('Nurse:Dentist ratio by size')
axes[0,0].legend(fontsize=8)

# Nurse:dentist by model
data = [df[df['archetype_model'] == m]['nurse_dentist_ratio'].dropna() for m in MODEL_ORDER]
bp = axes[0,1].boxplot(data, patch_artist=True, widths=0.5, notch=False,
                        medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'], palette_model):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[0,1].axhline(1.0, color=RED, lw=1.5, ls='--', label='1:1 benchmark')
axes[0,1].set_xticks(range(1, len(MODEL_ORDER)+1))
axes[0,1].set_xticklabels(MODEL_ORDER, rotation=12, ha='right')
axes[0,1].set_ylabel('Nurses per dentist')
axes[0,1].set_title('Nurse:Dentist ratio by model')
axes[0,1].legend(fontsize=8)

# Dentists per surgery by size
data = [df[df['archetype_size'] == s]['dentists_per_surgery'].dropna() for s in SIZE_ORDER]
bp = axes[1,0].boxplot(data, patch_artist=True, widths=0.5, notch=False,
                        medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'], palette_size):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[1,0].axhline(1.0, color=RED, lw=1.5, ls='--', label='1 dentist per surgery')
axes[1,0].set_xticks(range(1, len(SIZE_ORDER)+1))
axes[1,0].set_xticklabels(SIZE_ORDER, rotation=12, ha='right')
axes[1,0].set_ylabel('Dentists per surgery')
axes[1,0].set_title('Dentists per surgery by size')
axes[1,0].legend(fontsize=8)

# Dentists per surgery by model
data = [df[df['archetype_model'] == m]['dentists_per_surgery'].dropna() for m in MODEL_ORDER]
bp = axes[1,1].boxplot(data, patch_artist=True, widths=0.5, notch=False,
                        medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'], palette_model):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[1,1].axhline(1.0, color=RED, lw=1.5, ls='--', label='1 dentist per surgery')
axes[1,1].set_xticks(range(1, len(MODEL_ORDER)+1))
axes[1,1].set_xticklabels(MODEL_ORDER, rotation=12, ha='right')
axes[1,1].set_ylabel('Dentists per surgery')
axes[1,1].set_title('Dentists per surgery by model')
axes[1,1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 5 · 16-cell staffing heatmaps

The full 4×4 matrix view shows how staffing varies across *both* dimensions simultaneously.
This is the most actionable view for setting archetype-specific staffing targets.

In [ ]:
NA_ZONE_PAIRS = {('NHS Led', 'Large/Advanced'), ('NHS Led', 'Flagship')}

def matrix_heatmap(ax, df, metric_col, title, fmt='.1f', cmap='Blues'):
    pivot = (
        df.groupby(['archetype_size', 'archetype_model'], observed=True)[metric_col]
        .mean()
        .unstack(fill_value=np.nan)
        .reindex(index=SIZE_ORDER, columns=MODEL_ORDER)
    )
    annot = pivot.applymap(lambda v: f'{v:{fmt}}' if not np.isnan(v) else '')
    sns.heatmap(pivot, ax=ax, annot=annot, fmt='', cmap=cmap,
                linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.7})
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Model')
    ax.set_ylabel('Size')
    ax.tick_params(axis='x', rotation=20)
    ax.tick_params(axis='y', rotation=0)
    for model_val, size_val in NA_ZONE_PAIRS:
        ri = SIZE_ORDER.index(size_val)
        ci = MODEL_ORDER.index(model_val)
        ax.add_patch(plt.Rectangle((ci, ri), 1, 1, fill=True, color='#d0d0d0', zorder=3, alpha=0.8))
        ax.text(ci+0.5, ri+0.5, 'N/A', ha='center', va='center', fontsize=8, color='grey', zorder=4)

metrics = [
    ('position_dentist',      'Avg dentists',         '.1f', 'Blues'),
    ('position_dental_nurse', 'Avg dental nurses',    '.1f', 'Oranges'),
    ('position_hygienist',    'Avg hygienists',       '.2f', 'Greens'),
    ('position_receptionist', 'Avg receptionists',    '.1f', 'Reds'),
    ('total_headcount',       'Avg total headcount',  '.1f', 'Purples'),
    ('nurse_dentist_ratio',   'Nurse:Dentist ratio',  '.2f', 'YlOrBr'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('5 · Staffing Matrix — Size x Model', fontsize=13, fontweight='bold')

for ax, (col, title, fmt, cmap) in zip(axes.flat, metrics):
    matrix_heatmap(ax, df, col, title, fmt, cmap)

plt.tight_layout()
plt.show()

---
## 6 · Target staffing model

The table below shows the **recommended (median) staffing model** for each archetype
cell, expressed as headcount ranges (P25–P75) and the median contracted hours per role.
This is the foundation for archetype-specific job family frameworks and workforce
planning tools.

A practice whose staffing falls materially outside the range for its archetype is
either under-resourced or over-staffed relative to peers — both warranting review.

In [ ]:
def headcount_range(series):
    p25 = series.quantile(0.25)
    med = series.median()
    p75 = series.quantile(0.75)
    return f'{p25:.0f} – {med:.0f} – {p75:.0f}'

target = (
    df.groupby(['archetype_size', 'archetype_model'], observed=True)
    .agg(
        n=('practicekey', 'count'),
        dentists_range=('position_dentist',      headcount_range),
        nurses_range=('position_dental_nurse',   headcount_range),
        hygienists_range=('position_hygienist',  headcount_range),
        reception_range=('position_receptionist',headcount_range),
        total_range=('total_headcount',          headcount_range),
        med_dentist_hrs=('contractualhours_dentist',    'median'),
        med_nurse_hrs=('contractualhours_dental_nurse', 'median'),
        median_nurse_dentist=('nurse_dentist_ratio',    'median'),
    )
    .round(2)
)

print('Target staffing model (P25 - Median - P75 headcount ranges):')
display(target)

---
## 7 · Career mobility framework

Career mobility across the archetype matrix depends on which roles *exist* at each
archetype. The chart below visualises **role availability and scale** across all
archetypes, revealing natural career pathways:

- **Vertical mobility (size):** moving from Small/Foundation → Flagship opens up
  leadership roles (more complex Practice Manager responsibilities, clinical leads)
- **Horizontal mobility (model):** moving from NHS Led → Specialist/Referral Hub
  increases the importance of the Hygienist role and private patient management
- **Cross-dimensional moves:** e.g. a Dental Nurse in a Medium/Core NHS Led practice
  moving to a Small/Foundation Specialist practice to develop their specialist skill set

In [ ]:
# For each archetype, compute mean headcount per role — then visualise as bubble chart
arch_role = (
    df.groupby('archetype_rules')[ROLE_POS_COLS + ['archetype_size', 'archetype_model']]
    .agg({col: 'mean' for col in ROLE_POS_COLS} | {'archetype_size': 'first', 'archetype_model': 'first'})
    .reset_index()
)
arch_role.columns = ['archetype_rules'] + ROLES + ['archetype_size', 'archetype_model']

# Assign numeric positions for the grid
arch_role['x'] = arch_role['archetype_model'].map({m: i for i, m in enumerate(MODEL_ORDER)})
arch_role['y'] = arch_role['archetype_size'].map({s: i for i, s in enumerate(SIZE_ORDER)})

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('7 · Role Scale Across the Archetype Matrix\n(bubble size = average headcount)',
             fontsize=12, fontweight='bold')

for ax, role in zip(axes, ROLES):
    max_val = arch_role[role].max()
    for _, row in arch_role.iterrows():
        val = row[role]
        if val > 0:
            ax.scatter(row['x'], row['y'], s=val / max_val * 800,
                       color=ROLE_COLORS[role], alpha=0.7, edgecolors='white', lw=1)
            ax.text(row['x'], row['y'], f'{val:.1f}',
                    ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')
    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in MODEL_ORDER], fontsize=7)
    ax.set_yticks(range(len(SIZE_ORDER)))
    ax.set_yticklabels(SIZE_ORDER, fontsize=7)
    ax.set_xlim(-0.6, len(MODEL_ORDER) - 0.4)
    ax.set_ylim(-0.6, len(SIZE_ORDER) - 0.4)
    ax.set_title(role, color=ROLE_COLORS[role], fontweight='bold')
    ax.set_xlabel('Model', fontsize=8)
    if ax == axes[0]:
        ax.set_ylabel('Size', fontsize=8)
    # N/A zones
    for model_val, size_val in NA_ZONE_PAIRS:
        xi = MODEL_ORDER.index(model_val)
        yi = SIZE_ORDER.index(size_val)
        ax.add_patch(plt.Rectangle((xi-0.5, yi-0.5), 1, 1, fill=True, color='#d8d8d8', zorder=0))

plt.tight_layout()
plt.show()

---
## 8 · Career pathway summary

The table below describes the **typical career development opportunities** for each
role as a practice grows in size or shifts in model orientation. This is the narrative
layer on top of the quantitative analysis above.

In [ ]:
career_pathways = pd.DataFrame([
    {
        'Role': 'Dentist',
        'Small/Foundation': 'Generalist; broad NHS/private mix; close patient relationships',
        'Medium/Core': 'Develops special interests; takes on mentoring of junior dentists',
        'Large/Advanced': 'Clinical lead potential; specialist referral networks; complex cases',
        'Flagship': 'Principal/partner track; clinical director; business development',
        'NHS Led -> Specialist': 'Shift toward private fees; treatment planning complexity increases; specialist CPD'
    },
    {
        'Role': 'Dental Nurse',
        'Small/Foundation': 'Chair-side generalist; reception cover; patient liaison',
        'Medium/Core': 'Dedicated chair-side; begins specialist nursing CPD',
        'Large/Advanced': 'Specialist nursing (ortho, implants, sedation); mentoring juniors',
        'Flagship': 'Lead dental nurse; clinical governance; training coordination',
        'NHS Led -> Specialist': 'Specialist nursing skills become prerequisite; private patient experience valued'
    },
    {
        'Role': 'Hygienist',
        'Small/Foundation': 'Role typically absent or shared; greatest career opportunity in adding this role',
        'Medium/Core': 'Established role; builds recall book; supports private revenue growth',
        'Large/Advanced': 'Expanded scope; periodontal therapy; whitening/aesthetics',
        'Flagship': 'Senior hygienist; team lead; patient education programmes',
        'NHS Led -> Specialist': 'Key differentiator for private/specialist archetypes; low in NHS Led'
    },
    {
        'Role': 'Receptionist',
        'Small/Foundation': 'Multi-tasking; admin and patient-facing; practice heartbeat',
        'Medium/Core': 'Specialisation begins (treatment coordinator vs front desk)',
        'Large/Advanced': 'Treatment coordinator role emerges; revenue-generating capability',
        'Flagship': 'Head receptionist; TCO lead; patient journey management',
        'NHS Led -> Specialist': 'TCO role becomes central in private/specialist archetypes'
    },
    {
        'Role': 'Practice Manager',
        'Small/Foundation': 'Often dentist or senior nurse; operations + clinical governance',
        'Medium/Core': 'Dedicated PM; HR, compliance, KPI management',
        'Large/Advanced': 'Multi-site awareness; business performance; people management',
        'Flagship': 'Regional PM pathway; P&L accountability; strategic planning'
            + '; potential Area Manager progression',
        'NHS Led -> Specialist': 'Private revenue management and commercial skills increase in importance'
    },
])

display(
    career_pathways.set_index('Role')
    .style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap', 'font-size': '11px'})
    .set_table_styles([{'selector': 'th', 'props': [('text-align','left'), ('font-weight','bold')]}])
)

---
## 9 · Save staffing model output

In [ ]:
staffing_out = (
    df.groupby(['archetype_size', 'archetype_model', 'archetype_rules'], observed=True)
    .agg(
        n=('practicekey', 'count'),
        med_dentists=('position_dentist', 'median'),
        med_nurses=('position_dental_nurse', 'median'),
        med_hygienists=('position_hygienist', 'median'),
        med_receptionists=('position_receptionist', 'median'),
        med_total=('total_headcount', 'median'),
        med_dentist_hrs=('contractualhours_dentist', 'median'),
        med_nurse_hrs=('contractualhours_dental_nurse', 'median'),
        med_nurse_dentist_ratio=('nurse_dentist_ratio', 'median'),
        pct_with_hygienist=('has_hygienist', 'mean'),
    )
    .round(2)
    .reset_index()
)
staffing_out['pct_with_hygienist'] = (staffing_out['pct_with_hygienist'] * 100).round(1)
staffing_out.to_csv('staffing_by_archetype.csv', index=False)
print(f'Saved staffing_by_archetype.csv  ({len(staffing_out)} archetype cells)')
display(staffing_out)